# `population` 02: noteworthy single-feature findings

**Purpose:** identify and discuss supported points that stand out after the
standard `numeric` breakdown. Target relationships here are
exploratory and must be rechecked after the split is frozen.


In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def find_stage_directory():
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (
            (candidate / "data" / "TrainingSetValues.csv").exists()
            and (candidate / "src" / "source_data_validation.py").exists()
        ):
            return candidate
    raise FileNotFoundError("Could not locate the stage-1-pump-it-up directory.")


stage_directory = find_stage_directory()
source_directory = str((stage_directory / "src").resolve())
if source_directory not in sys.path:
    sys.path.insert(0, source_directory)

from predictor_audit import (
    analysis_categories,
    categorical_summary,
    categorical_target_profile,
    category_frequency_table,
    numeric_summary,
    numeric_target_summary,
    related_feature_summary,
    sentinel_mask,
    source_blank_mask,
    text_normalisation_summary,
)
from source_data_validation import (
    validate_aligned_ids,
    validate_label_frame,
    validate_raw_feature_schema,
)

data_directory = stage_directory / "data"
training_features = pd.read_csv(
    data_directory / "TrainingSetValues.csv",
    keep_default_na=False,
)
training_labels = pd.read_csv(
    data_directory / "TrainingSetLabels.csv",
    keep_default_na=False,
)
test_features = pd.read_csv(
    data_directory / "TestSetValues.csv",
    keep_default_na=False,
)

validate_raw_feature_schema(training_features)
validate_raw_feature_schema(test_features)
validate_label_frame(training_labels)
validate_aligned_ids(training_features, training_labels)

training_data = training_features.merge(
    training_labels,
    on="id",
    validate="one_to_one",
)

feature = 'population'
feature_metadata = {'order': 17, 'name': 'population', 'audit_type': 'numeric', 'role': 'candidate', 'disposition': 'retain zero, one and positive magnitude as distinct states', 'finding': 'Zero and one are large special spikes; positive values are strongly right-skewed.', 'decision': 'Keep zero and one flags plus a raw or log1p magnitude candidate.', 'risk': 'Zero is probably often missing while one may be a recorded placeholder or real small population.', 'sentinel_values': [0], 'related': [{'feature': 'gps_height', 'reason': 'Population and height zeros frequently occur in one measurement block.'}, {'feature': 'region', 'reason': 'Population distribution varies geographically.'}, {'feature': 'lga', 'reason': 'Local administrative context may explain population scale.'}]}
feature_types = {'amount_tsh': 'numeric', 'date_recorded': 'date', 'funder': 'high-cardinality-category', 'gps_height': 'numeric', 'installer': 'high-cardinality-category', 'longitude': 'coordinate', 'latitude': 'coordinate', 'wpt_name': 'high-cardinality-category', 'num_private': 'numeric', 'basin': 'category', 'subvillage': 'high-cardinality-category', 'region': 'category', 'region_code': 'category', 'district_code': 'category', 'lga': 'category', 'ward': 'high-cardinality-category', 'population': 'numeric', 'public_meeting': 'binary', 'recorded_by': 'constant', 'scheme_management': 'category', 'scheme_name': 'high-cardinality-category', 'permit': 'binary', 'construction_year': 'year', 'extraction_type': 'category', 'extraction_type_group': 'category', 'extraction_type_class': 'category', 'management': 'category', 'management_group': 'category', 'payment': 'category', 'payment_type': 'category', 'water_quality': 'category', 'quality_group': 'category', 'quantity': 'category', 'quantity_group': 'category', 'source': 'category', 'source_type': 'category', 'source_class': 'category', 'waterpoint_type': 'category', 'waterpoint_type_group': 'category'}
assert feature in training_features.columns
print(
    f"Validated {len(training_features):,} training rows and "
    f"{len(test_features):,} test rows for {feature}."
)


Validated 59,400 training rows and 14,850 test rows for population.


## Supported target evidence


In [2]:
sentinel_values = [0]
display(
    numeric_target_summary(
        training_data,
        feature,
        sentinel_values=sentinel_values,
    )
)

values = pd.to_numeric(training_data[feature], errors="coerce")
state = pd.Series("observed", index=training_data.index, dtype="string")
state.loc[values.isna()] = "missing"
state.loc[values.isin(sentinel_values)] = "configured sentinel"
state_profile = pd.crosstab(
    state,
    training_data["status_group"],
    normalize="index",
).mul(100).round(2)
state_profile.insert(0, "rows", state.value_counts())
display(state_profile)


,rows,missing,median,mean,sentinel rows,zero rows (%),p90
status_group,,,,,,,
functional,32259,0,40.0,187.553,11274,34.948,460.0
functional needs repair,4317,0,25.0,175.102,1775,41.117,500.0
non functional,22824,0,1.0,170.016,8332,36.505,450.0


status_group,rows,functional,functional needs repair,non functional
row_0,,,,
configured sentinel,21381,52.73,8.30,38.97
observed,38019,55.20,6.69,38.12


## Observation

Zero and one are large special spikes; positive values are strongly right-skewed.

## Interpretation

The supported single-feature patterns make this field worth the stated
treatment, but they do not prove causation or independent predictive value.
High-cardinality and geographic fields are especially vulnerable to
memorisation under a random split.

## Provisional decision

Keep zero and one flags plus a raw or log1p magnitude candidate.

**Risk to carry forward:** Zero is probably often missing while one may be a recorded placeholder or real small population.


In [3]:
decision_record = pd.DataFrame([{
    "feature": feature,
    "role": feature_metadata["role"],
    "disposition": feature_metadata["disposition"],
    "finding": feature_metadata["finding"],
    "decision": feature_metadata["decision"],
    "risk": feature_metadata["risk"],
}])
display(decision_record.set_index("feature"))


,role,disposition,finding,decision,risk
feature,,,,,
population,candidate,"retain zero, one and positive magnitude as dis...",Zero and one are large special spikes; positiv...,Keep zero and one flags plus a raw or log1p ma...,Zero is probably often missing while one may b...
